# How to use Training Wrappers

We have just spent some time understanding how to create the steps of an end-to-end training, prediction and validation pipeline using PyEarthTools and PyTorch. It should be noted, than any machine learning framework could be used - the data loaders and pipelines in PyEarthTools can (and have) been used with other machine learning frameworks.

It also starts to become clearer how much detail there is in the actual training process, separately from defining pre-processing and model architecture. Concerns dealt with during the training process include:

1. Stopping early to check training is 'on track'
2. Selecting a batch size for optimal training performance
3. Loading data quickly enough to keep the GPU busy
4. Not overloading GPU memory
5. Saving and loading weights to avoid retraining every time
6. Saving model weights alongside their configuration (to avoid 'mystery weights' with no clear provenance)
7. Choosing hyperparameters, like epochs, number of samples, date-ranges and batch sizes
8. Logging performance during training
9. Choosing good filenames for things

These things can all be done in PyTorch, but then there is a lot of bespoke code per-model which has to be invented by each development team. Also, these choices form part of the sources of non-reproducibility between models. By standardising some of these practises, the effort involved in setting up a sensible and repeatable training process can be reduced. PyEarthTools provides a the following things to assist with this:

1. Simple pre-defined model architectures which can be quickly adopted
2. Standard classes for training which will take care of many of these decisions
3. Integration with standard tools that will be familiar, but reducing the cost of adopting them
4. Integration with PyTorch Lightning for a structured approach to training management
5. Allowing a rollout period on autoregressive models (recurrent models)

This tutorial aims to showcase some of these classes. We will re-do the first prediction architecture, this time using PyEarthTools standard classes. We're going to imagine we want to use [PyTorch Lightning](https://lightning.ai/docs/pytorch/stable/) for the underlying model architecture, but integrate with PyEarthtools from the outset.

## Starting Simple

Doing everything on this list is a lot to tackle at once. Indeed, PyEarthTools has incomplete support for some of these components. We will tackle the following goals:

1. How to put a model in a pipeline for inference
2. How to keep the "top n" weights during training
3. How to monitor validation loss during training

The simplest first step is to wrap the model architecture from the previous tutorial in a SimpleModel wrapper. This is the quickest way to achieve the first goal. We will then add the "top n" strategy and keep a log of validation loss.

In [1]:
# Core Python imports
import datetime
import functools
import os
import pathlib
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Scientific standard imports
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

In [3]:
# PyEarthTools imports including NCI cached data
import pyearthtools.data as petdata
import pyearthtools.pipeline as petpipe
from pyearthtools.data.time import Petdt

my_site = 'site_archive_nci'  # set this to 'site_archive_nci', 'site_archive_jasmin' or 'site_archive_met_office'
import importlib
_ = importlib.import_module(my_site)

In [4]:
# PyTorch imports
import torch
import torch.nn as nn
import torch.optim as optim

# PyTorch Lightning Imports
import lightning as L

# Set random seed for reproducibility
torch.manual_seed(42)

# Autodetect GPU and use if possible
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Detected device {device}, will attempt to use for training")

Detected device cuda:0, will attempt to use for training


## Load and prepare data

We will jump straight to the nearly same pipeline as previous examples.

In [5]:
# We specify the date, hour, and minute for querying data
selected_date = '20210609T02:00'

In [6]:
himawari = petdata.archive.Himawari('surface_global_irradiance')

# We will try to predict one-hour jumps
temporal_window = petpipe.modifications.TemporalWindow(
    prior_indexes=[-1], 
    posterior_indexes=[0], 
    merge_method = functools.partial(xr.concat, dim='time'),
    timedelta=petdata.time.TimeDelta((20, "minutes"))
)

In [7]:
xarray_steps = petpipe.Pipeline(
    himawari,
    petpipe.operations.xarray.Sort(order=['time', 'latitude', 'longitude']),  # 
    # Align the data variable's coordinate order to the dataset coordinate order so all arrays are the same shape
    petpipe.operations.xarray.AlignDataVariableDimensionsToDatasetCoords(),  
    petdata.transform.region.Bounding(-35, -25, 138, 150),  # cut down on region for example
    petpipe.operations.xarray.normalisation.SingleValueDivision(1200),
    temporal_window,
    exceptions_to_ignore=petdata.exceptions.DataNotFoundError    
)

numpy_conversion = petpipe.Pipeline(
    petpipe.operations.xarray.conversion.ToNumpy(),
    petpipe.operations.numpy.reshape.Rearrange('c t h w -> t c h w'), # channel time height width -> time channel height width
    exceptions_to_ignore=petdata.exceptions.DataNotFoundError
)

fullsat = xarray_steps + numpy_conversion
# fullsat

In [8]:
train_split = petpipe.iterators.DateRange("20200101T00", "20210101T00", interval="20 minutes")
valid_split = petpipe.iterators.DateRange("20210201T00", "20210501T00", interval="20 minutes")

batch_size = 8
num_workers = 0
max_epochs = 3

### Define the Prediction architecture
We still have a PyTorch model to define. You can use a custom architecture, or a PyEarthTools "Simple" model (coming soon thanks to work in progress at Earth Sciences New Zealand), or an advanced architecture such as FourCastNet or LUCIE. For now, we will stick with the same model for familiarity, and then wrap it into PyEarthTools wrapper classes to tap into more advanced training approached.

In [9]:
# Reminder, the image size is latitude: 1726, longitude: 2214

class EncoderDecoder(nn.Module):
    def __init__(self, 
                 input_height = 501,
                 input_width = 601,
                 kernel_size = 4,
                 stride=2,
                 input_channel_count = 2,
                 output_channel_count = 2,
                 latent_dim=300):
        super(EncoderDecoder, self).__init__()

        self.input_width = input_width
        self.input_height = input_height
        self.input_channel_count = input_channel_count
        self.output_channel_count = output_channel_count

        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels=self.input_channel_count, out_channels=16, kernel_size=kernel_size, stride = stride, padding = 1),
            nn.ReLU(),
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride =stride, padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=7),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(in_channels=64, out_channels=32, kernel_size=7),
            nn.ReLU(),
            nn.ConvTranspose2d(in_channels=32, out_channels=16, kernel_size=3, stride=stride, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(in_channels=16, out_channels=self.output_channel_count, kernel_size=kernel_size, stride=stride, padding=1, output_padding=1),
            nn.Sigmoid()
        )
        

    def forward(self, x):

        # Get latent representation
        latent = self.encoder(x)

        # Reconstruct input
        reconstructed = self.decoder(latent)

        return reconstructed

In [10]:
class LightningWrapper(L.LightningModule):
    def __init__(self, model, lr=1e-3):
        super().__init__()
        self.model = model
        self.lr = lr
        self.criterion = nn.L1Loss()

    def forward(self, x):
        return self.model(x)

    def _shared_step(self, batch, stage):
        '''
        Calculate and return loss, standardised across training and validation
        '''
        x, y = batch
        X = self(x.squeeze(axis=1).float())
        loss = self.criterion(X, y)
        self.log(f"{stage}_loss", loss, prog_bar=True, on_epoch=True, on_step=False)
        return loss        

    def training_step(self, batch, batch_idx):
        return self._shared_step(batch, "train")

    def validation_step(self, batch, batch_idx):
        return self._shared_step(batch, "validate")

    def predict_step(self, batch, batch_idx):
        return self(batch)

    def configure_optimizers(self):
        return optim.Adam(self.parameters(), lr=self.lr)

In [11]:
base_model = EncoderDecoder(input_channel_count=1, output_channel_count=1).to(device)  
lightning_model = LightningWrapper(base_model)

In [12]:
import pyearthtools.training
data_module = pyearthtools.training.data.lightning.PipelineLightningDataModule(
    fullsat,                  # Data pipeline (no iterator required)
    train_split=train_split,  # Iterator for the training split
    valid_split=valid_split,  # Iterator for the validation split
    batch_size=batch_size,    # Batch size
    num_workers=num_workers,  # Number of PyTorch workers
    iterator_dataset=True
)

data_module

PipelineLightningDataModule
	Initialisation                 Pytorch Lightning DataModule.
		 batch_size                     8
		 iterator_dataset               True
		 num_workers                    0
		 pipelines                      {'Pipeline': {'__args': '(Himawari\n\tDescription                    Himawari 8/9 satellite data\n\t\t Range                          \'2019-current\'\n\t\t Resolution                     \'10 minutes\'\n\n\n\tInitialisation                 \n\t\t data_interval                  (10, \'m\')\n\t\t file_regex                     \'*{date_info}*{time_info}*.nc\'\n\t\t variables                      [\'surface_global_irradiance\']\n\tTransforms                     \n\t\t StandardCoordinateNames        {\'latitude\': "[\'lat\', \'Latitude\', \'yt_ocean\', \'yt\']", \'longitude\': "[\'lon\', \'Longitude\', \'xt_ocean\', \'xt\']", \'replacement_dictionary\': \'None\', \'time\': "[\'Time\']"}\n\t\t Trim                           {\'__args\': \'()\', \'variables\': "[\'surface_global_irradiance\']"}, Sort\n\tInitialisation                 Sort Variables of an `xarray` object\n\t\t order                          [\'time\', \'latitude\', \'longitude\']\n\t\t strict                         False, AlignDataVariableDimensionsToDatasetCoords\n\tInitialisation                 Sometimes, the data variables within a dataset may not be ordered consistently., Bounding\n\tInitialisation                 Cut with Bounding box\n\t\t max_lat                        -25\n\t\t max_lon                        150\n\t\t min_lat                        -35\n\t\t min_lon                        138, SingleValueDivision\n\tInitialisation                 Division based Normalisation\n\t\t division_factor                1200, TemporalWindow\n\tInitialisation                 The purpose of this class is to provide the ability to perform, ToNumpy\n\tInitialisation                 Convert xarray objects to np.ndarray\'s\n\t\t reference_dataset              None\n\t\t run_parallel                   False\n\t\t saved_records                  None\n\t\t warn                           True, Rearrange\n\tInitialisation                 Operation to rearrange data using einops\n\t\t rearrange                      \'c t h w -> t c h w\'\n\t\t rearrange_kwargs               None\n\t\t reverse_rearrange              None\n\t\t skip                           False)', 'exceptions_to_ignore': "<class 'pyearthtools.data.exceptions.DataNotFoundError'>", 'iterator': 'None', 'max_exception_count': '-1', 'name': 'None', 'sampler': 'None'}}
		 train_split                    {'DateRange': {'allowlist': 'None', 'blocklist': 'None', 'end': "'20210101T00'", 'interval': "'20 minutes'", 'start': "'20200101T00'"}}
		 valid_split                    {'DateRange': {'allowlist': 'None', 'blocklist': 'None', 'end': "'20210501T00'", 'interval': "'20 minutes'", 'start': "'20210201T00'"}}

In [ ]:
import pyearthtools.training.wrapper.lightning
workdir = '~/pet_training_cache'  # Set this to somewhere you're happy to put your training experiments
trainer = pyearthtools.training.lightning.Train(
    lightning_model,
    data_module,
    path=workdir,
    trainer_kwargs={
        "max_epochs": max_epochs,
        "devices": 1,
        "num_sanity_val_steps": 0,
        "logger": False,
        "enable_checkpointing": True,
        "enable_model_summary": False,
    },
)

trainer.fit(load=False)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

In [ ]:
print(os.listdir(pathlib.Path(workdir).expanduser() / 'checkpoints'))